# 03b · 5.ª familia: modelo preentrenado (embeddings) y comparación de candidatos
Se usa un modelo de oraciones multilingüe **preentrenado** (no se ajusta con nuestros datos, así que no hay fuga) y sobre sus embeddings una regresión logística. Misma validación y métrica que los demás.
Ejecutar después de `03`. La primera vez descarga el modelo y calcula los embeddings (varios minutos en CPU).

In [1]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from sklearn.linear_model import LogisticRegression
from src.datos import cargar_obt, cargar_obt_entrenamiento
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt_entrenamiento())
train, test = split_temporal(df, 0.2)
X, y = train[FEATURES], train[TARGET]
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


In [2]:
limpiar_fase("candidato", familias=["estado_del_arte"])
r = correr("candidato-Embeddings+LogReg",
           construir_pipeline(LogisticRegression(max_iter=1000), texto="embeddings"),
           "candidato", "estado_del_arte", X, y,
           {"modelo": "LogisticRegression", "embeddings": EmbeddingTransformer().modelo, "max_iter": 1000}, FECHA_CORTE)
pd.Series({k: v for k, v in r.items() if k not in ('run_id', 'nombre', 'model_version')}).round(4)

Loading weights:   0%|                                                                                                     | 0/199 [00:00<?, ?it/s]

Loading weights:   2%|█▊                                                                                           | 4/199 [00:00<00:08, 24.20it/s]

Loading weights:  19%|█████████████████▍                                                                         | 38/199 [00:00<00:01, 135.33it/s]

Loading weights:  38%|██████████████████████████████████▊                                                        | 76/199 [00:00<00:00, 196.10it/s]

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 370.49it/s]

/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/11/24 18:55:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run candidato-Embeddings+LogReg at: http://mlflow:5000/#/experiments/2/runs/479877ebacc24297b4492aa14cae1bce
🧪 View experiment at: http://mlflow:5000/#/experiments/2


train_pr_auc               0.9552
cv_pr_auc_mean             0.9461
cv_pr_auc_std              0.0145
brecha_pr_auc              0.0091
cv_roc_auc_mean            0.9601
cv_f1_mean                 0.9122
cv_f1_std                  0.0241
umbral_costo               0.2200
costo_cv_umbral            0.1284
costo_cv_050               0.1838
segundos_entrenamiento     1.7221
segundos_cv_por_pliegue    8.3982
dtype: float64

## Comparación de todos los candidatos (leída desde MLflow) y elección de los 3 a ajustar

In [3]:
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
runs = mlflow.search_runs([exp.experiment_id])
runs = runs[runs["tags.fase"].isin(["baseline", "candidato"])]
t = runs.set_index("tags.mlflow.runName")[["tags.fase","tags.familia","metrics.train_pr_auc","metrics.cv_pr_auc_mean","metrics.cv_pr_auc_std","metrics.brecha_pr_auc","metrics.costo_cv_umbral","metrics.segundos_entrenamiento"]]
t.columns = ["fase","familia","train_pr_auc","cv_pr_auc_mean","cv_pr_auc_std","brecha","costo_cv","seg_entren"]
t = t.sort_values("cv_pr_auc_mean", ascending=False).round(4)
display(t)

,fase,familia,train_pr_auc,cv_pr_auc_mean,cv_pr_auc_std,brecha,costo_cv,seg_entren
tags.mlflow.runName,,,,,,,,
candidato-Embeddings+LogReg,candidato,estado_del_arte,0.9552,0.9461,0.0145,0.0091,0.1284,1.7221
candidato-LogisticRegression,candidato,lineal_regularizado,0.9667,0.9434,0.0176,0.0234,0.1559,0.9985
candidato-LightGBM,candidato,boosting,0.9851,0.9405,0.0150,0.0447,0.1562,1.7560
candidato-RandomForest,candidato,bagging,0.9972,0.9282,0.0213,0.0689,0.1728,2.2150
candidato-MLPClassifier,candidato,red_neuronal,0.9992,0.9191,0.0290,0.0801,0.2342,53.2844
baseline-regla-lexico,baseline,regla_negocio,0.7494,0.7488,0.0662,0.0006,0.7870,0.0003
baseline-dummy,baseline,dummy,0.4642,0.4700,0.1137,-0.0059,0.5300,0.0013
